# Tutorial 3: Live Waveform & Spectral Streaming

This tutorial demonstrates real-time data access via WebSocket connections
to the EQ Synapse gateway.

**What you will learn:**
1. Connect to the spectral WebSocket for live FFT data
2. Reconfigure the spectral stream (phase, FFT size)
3. Connect to the CPOW waveform WebSocket for raw sample data
4. Capture a snapshot of live waveform data to a parquet file

**Prerequisites:**
- A running EQ Synapse gateway at `http://localhost:8080`
- The `equser[analysis]` package (`pip install equser[analysis]`)

**Data formats:**
- **Spectral stream**: JSON messages with FFT magnitude and frequency arrays
- **CPOW stream**: Arrow IPC binary messages (~512 rows / 16 ms per message at 32 kHz)

## 1. Setup

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.parquet as pq
import websocket
%matplotlib inline

from equser.api import SynapseClient, connect_cpow_stream, connect_spectral_stream

GATEWAY = 'http://localhost:8080'
client = SynapseClient(GATEWAY)

In [ ]:
# Discover devices so we can pick a device_id
devices = client.list_devices()
device_id = devices[0]['id'] if devices else 'wave-001'
print(f"Using device: {device_id}")

## 2. Spectral streaming

The spectral WebSocket at `/api/ws/spectral` delivers real-time FFT frames as JSON.
Each frame contains the magnitude spectrum for a single channel.

**Query parameters:**
- `device_id` (required)
- `phase`: channel to analyze (`va`, `vb`, `vc`, `ia`, `ib`, `ic`; default `va`)
- `fft_size`: window size (default 4096, must be power of 2)
- `update_rate`: frames per second (default 10.0)
- `freq_min`, `freq_max`: frequency range in Hz (defaults 0–3000)

In [ ]:
# Collect a few spectral frames
frames = []
max_frames = 5

print(f"Collecting {max_frames} spectral frames...")
for i, frame in enumerate(connect_spectral_stream(device_id, phase='va', fft_size=4096, gateway_url=GATEWAY)):
    frames.append(frame)
    if i + 1 >= max_frames:
        break

print(f"Received {len(frames)} frames")
if frames:
    print(f"Frame keys: {list(frames[0].keys())}")

In [ ]:
# Plot the latest spectrum
if frames:
    frame = frames[-1]
    # Frame keys depend on the server version; try common variants.
    freqs = np.array(frame.get('frequencies', frame.get('freq', [])))
    mags = np.array(frame.get('magnitudes', frame.get('magnitude', [])))

    if len(freqs) > 0 and len(mags) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.set_title('Live Spectral Snapshot (Phase VA)')
        ax.plot(freqs, mags, 'k-', linewidth=0.5)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('Magnitude')
        ax.set_xlim(0, 3000)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print("Frame data:")
        print(json.dumps(frame, indent=2, default=str))

## 3. Reconfigure the spectral stream

You can send a JSON configuration message to change the channel, FFT size,
or update rate without reconnecting.

In [ ]:
# Connect, reconfigure, and collect frames from a different phase
ws_url = GATEWAY.replace('http://', 'ws://') + f'/api/ws/spectral?device_id={device_id}&phase=va'
ws = websocket.create_connection(ws_url, timeout=10)

# Reconfigure to phase VB with a larger FFT window
config_msg = json.dumps({
    'type': 'config',
    'phase': 'vb',
    'fft_size': 8192,
    'update_rate': 5.0
})
ws.send(config_msg)
print(f"Sent config update: {config_msg}")

# Collect a few frames with the new config
reconfigured_frames = []
for _ in range(3):
    opcode, data = ws.recv_data()
    if opcode == websocket.ABNF.OPCODE_TEXT:
        reconfigured_frames.append(json.loads(data.decode('utf-8')))

ws.close()
print(f"Received {len(reconfigured_frames)} frames after reconfiguration")

## 4. CPOW waveform streaming

The CPOW WebSocket at `/api/ws/cpow_stream` delivers raw waveform samples as
Arrow IPC binary messages. Each message contains ~512 rows (16 ms at 32 kHz).

Text messages are JSON gap markers (`{"type": "gap", "skipped_samples": N}`)
indicating that the reader fell behind the real-time stream.

In [ ]:
# Collect ~100 ms of live waveform data (about 6-7 messages)
batches = []
gaps = []
target_samples = 3200  # 100 ms at 32 kHz
total_samples = 0

print("Collecting live waveform data...")
for msg in connect_cpow_stream(gateway_url=GATEWAY):
    if isinstance(msg, dict):
        # Gap marker
        gaps.append(msg)
        print(f"  Gap: {msg.get('skipped_samples', '?')} samples skipped")
    else:
        # Arrow RecordBatch
        batches.append(msg)
        total_samples += len(msg)
        if total_samples >= target_samples:
            break

print(f"Collected {len(batches)} batches, {total_samples} total samples ({total_samples/32000*1000:.0f} ms)")
if gaps:
    print(f"Encountered {len(gaps)} gap(s)")

In [ ]:
# Concatenate batches into a single table and plot
if batches:
    combined = pa.Table.from_batches(batches)
    df = combined.to_pandas()
    n = min(3200, len(df))
    time_ms = np.arange(n) / 32.0

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title('Live Waveform Snapshot')
    for col, color in [('VA', 'black'), ('VB', 'red'), ('VC', 'blue')]:
        if col in df.columns:
            ax.plot(time_ms, df[col].values[:n], color=color, label=col, alpha=0.8, linewidth=0.5)
    ax.set_xlabel('Elapsed time (ms)')
    ax.set_ylabel('Voltage')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Snapshot capture

Capture a few seconds of live waveform data and save it to a local parquet file
for offline analysis.

> **Tip:** For longer captures or automated use, run `equser snapshot` from the command line.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

capture_duration_sec = 2
target_samples = capture_duration_sec * 32000

batches = []
total_samples = 0

print(f"Capturing {capture_duration_sec} seconds of live data...")
for msg in connect_cpow_stream(gateway_url=GATEWAY):
    if isinstance(msg, dict):
        continue  # Skip gap markers
    batches.append(msg)
    total_samples += len(msg)
    if total_samples >= target_samples:
        break

if batches:
    combined = pa.Table.from_batches(batches)
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    output_path = Path(f'snapshot_{timestamp}.parquet')
    pq.write_table(combined, output_path)
    print(f"Saved {len(combined)} samples to {output_path}")
    print(f"  Duration: {len(combined)/32000:.2f} seconds")
    print(f"  Size: {output_path.stat().st_size / 1024:.0f} KB")
else:
    print("No data received.")

## Next steps

- **Tutorial 1** (`01-parquet-files.ipynb`): Work with parquet files directly for offline analysis.
- **Tutorial 2** (`02-backend-api.ipynb`): Query historical data through the REST API.
- **Harmonic analysis** (`../analysis/harmonic-analysis.ipynb`): FFT-based analysis of CPOW data.
- **Snapshot tool**: Run `equser snapshot` from the command line for automated waveform capture.